In [20]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq


In [10]:
from dotenv import load_dotenv
load_dotenv()

import os
http_proxy = os.getenv('http_proxy')
https_proxy = os.getenv('https_proxy')
HTTP_PROXY = os.getenv('HTTP_PROXY')
HTTPS_PROXY = os.getenv('HTTPS_PROXY')



In [11]:
# Load keys from .env file
groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv("HF_TOKEN")

In [12]:
def load_pdfs(folder_path):
    loader = PyPDFDirectoryLoader(folder_path)
    documents = loader.load()
    return documents


pdf_folder = "/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/"

docs = load_pdfs(pdf_folder)

print("Total pages loaded:", len(docs))
print("Sample metadata:", docs[0].metadata)


Total pages loaded: 2320
Sample metadata: {'producer': 'Antenna House PDF Output Library 7.1.1639', 'creator': 'AH CSS Formatter V7.1 MR2 for Linux64 : 7.1.3.50324 (2021-04-26T09:47+09)', 'creationdate': '2024-11-01T20:52:54+00:00', 'author': 'John Berryman;Albert Ziegler;', 'moddate': '2024-11-06T01:48:50-05:00', 'title': 'Prompt Engineering for LLMs', 'ebx_publisher': "O'Reilly Media", 'source': '/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/Prompt_engineerng_llms.pdf', 'total_pages': 282, 'page': 0, 'page_label': 'Cover'}


In [13]:
def chunk_documents(docs, chunk_size=1400, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = splitter.split_documents(docs)
    return chunks


documents = chunk_documents(docs)

print("Total chunks:", len(documents))

Total chunks: 4269


In [14]:
clean_docs = [
    d for d in documents
    if isinstance(d.page_content, str) and len(d.page_content.strip()) > 0
]

print("Valid chunks:", len(clean_docs))


Valid chunks: 4269


In [17]:
def sanitize_documents(docs):
    cleaned = []

    for d in docs:
        text = str(d.page_content).strip()

        if len(text) < 20:   # remove tiny chunks
            continue

        d.page_content = text
        cleaned.append(d)

    return cleaned


clean_docs = sanitize_documents(documents)

print("Final usable chunks:", len(clean_docs))


Final usable chunks: 4268


In [15]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding size:", len(embeddings.embed_query("test")))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 486.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding size: 384
